# Benthic validation of the spin-up runs against SIBES and SUBES

Compares the modelled benthic feeding groups of each spin-up run with the SIBES
(intertidal) and SUBES (subtidal) macrozoobenthos surveys, both expressed in
g C m⁻².

| Model | Observation column | Feeding group |
|---|---|---|
| `Y1c` | `epibenthos_Y1_c` | epibenthos |
| `Y2c` | `deposit_feeders_Y2_c` | deposit feeders |
| `Y3c + Yy3c` | `suspension_feeders_Y3_c` | suspension feeders (adult + young) |
| `Y5c` | `endobenthos_Y5_c` | endobenthos / predators |

**Matching.** Every sample is paired with the nearest *wet* model cell, measured
in kilometres on a local grid (not in degrees, which at 53° N would favour
cells to the east or west), and with the nearest model day. Samples further than
`MAX_MATCH_KM` from any wet cell lie outside the model domain; they are left
unmatched instead of being snapped to a distant cell.

**Read before interpreting.**

- Only observation years present in the model output are matched. For a 2015
  spin-up that is SIBES 2015 (about 3,700 samples) and only a handful of SUBES
  samples, so SUBES skill is not meaningful until more model years exist.
- Benthic biomass is patchy and strongly right-skewed (most epibenthos samples
  are zero), so point-to-point skill is inherently low. Scatter and error axes
  are symmetric-log: linear below `LINTHRESH` so that zeros stay visible,
  logarithmic above.

**Outputs** (in `OUT_DIR/figures/`, each as vector PDF + 600 dpi PNG + CSV of the
plotted numbers):

| File | Content |
|---|---|
| `fig01_benthos_scatter` | modelled vs observed biomass, per run and feeding group |
| `fig02_benthos_error_distribution` | distribution of model − observed |
| `fig03_benthos_bias_map_<var>` | station bias and model/observed ratio, per feeding group |
| `fig04_benthos_skill_summary` | mean biomass, correlation and normalised RMSE per run |
| `fig05_benthos_distribution` | cumulative distributions, observed vs modelled |
| `fig06_benthos_depth_classes` | total biomass and feeding-group composition by depth below MSL |

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from scipy.stats import pearsonr, spearmanr

sys.path.insert(0, str(Path.cwd()))          # figstyle.py sits next to this notebook
import figstyle as fs

fs.use_style()

---
## 1. Configuration

The only cell you should normally need to edit.

In [ ]:
# --- paths -------------------------------------------------------------------
POSTPROC_DIR    = Path("/export/lv9/projects/dws/results/validation/benthos/")
BASE_OUTPUT_DIR = Path("/export/lv9/projects/dws/model_output/archived_runs/effective_fetch/")
OUT_DIR = POSTPROC_DIR / BASE_OUTPUT_DIR.name      # collocated tables, skill table
FIG_DIR = OUT_DIR / "figures"                      # figures + their CSV twins

OBS_CSV = {
    "SIBES": POSTPROC_DIR / "SIBES_v4_feeding_groups_gC_m2.csv",
    "SUBES": POSTPROC_DIR / "feeding_groups_gC_m2_ecotopes_SUBES.csv",
}
SPINUP_NAMES = ["spinup_01", "spinup_02"]
NC_PATTERN = "dws_500m.3d.{year}*.nc"   # {year} = each observation year

# True: reload the collocated tables written by an earlier run instead of
# re-reading the model output (fast when only the figures change).
REUSE_COLLOCATED = False

# --- variables ---------------------------------------------------------------
VALIDATION_VARS = ["Y1c", "Y2c", "Y3c", "Y5c"]
OBS_COL = {
    "Y1c": "epibenthos_Y1_c",
    "Y2c": "deposit_feeders_Y2_c",
    "Y3c": "suspension_feeders_Y3_c",
    "Y5c": "endobenthos_Y5_c",
}
GROUP_LABEL = {
    "Y1c": "Epibenthos",
    "Y2c": "Deposit feeders",
    "Y3c": "Suspension feeders",
    "Y5c": "Endobenthos",
}
MODEL_VARS = ["Y1c", "Y2c", "Y3c", "Yy3c", "Y5c"]   # read from the files
MGC_TO_GC = 1.0 / 1000.0                            # model mg C m-2 -> g C m-2

# --- grid / matching ---------------------------------------------------------
LON_VALID, LAT_VALID = (-180.0, 180.0), (-90.0, 90.0)   # lonc/latc fill is -999
BATHY_FILL = -10.0      # bathymetry missing_value marks land cells
MAX_MATCH_KM = 1.0      # 2 grid cells; further = outside the model domain
BASIN_CANONICAL = {"Eijerlandse Gat": "Eierlandse Gat"}  # SUBES spelling -> SIBES

# --- statistics and plotting -------------------------------------------------
SOURCES = ["SIBES", "SUBES"]
MIN_N = 30              # fewer pairs than this: no skill score, no histogram
LINTHRESH = 0.01        # g C m-2; symlog axes are linear below this
# Depth classes for Fig. 6: model depth below MSL at the matched cell, in m
# (bathymetry is positive down; negative = bed above MSL). Classes holding fewer
# than MIN_N samples are left out of the figure.
DEPTH_EDGES = [-np.inf, -0.5, 0.0, 0.5, 1.0, 1.5, 2.0, np.inf]
UNIT = r"g C m$^{-2}$"
SPINUP_COLOURS = fs.run_colours(SPINUP_NAMES)

FIG_DIR.mkdir(parents=True, exist_ok=True)
print("model runs :", ", ".join(f"{s} ({'found' if (BASE_OUTPUT_DIR / s).is_dir() else 'MISSING'})"
                             for s in SPINUP_NAMES))
print("figures to :", FIG_DIR)

---
## 2. Collocation helpers

`load_model_year` masks both fill values GETM writes into these averaged
fields: the declared `-9999`, and the `-9998` that fills the whole first frame
of every monthly file (there is no interval to average over yet). Month
boundaries occur in two files; the copy with valid data is kept.

In [ ]:
R_EARTH_KM = 6371.0


def load_observations() -> pd.DataFrame:
    """SIBES + SUBES in one table, with harmonised basin names and a year column."""
    frames = []
    for source, csv_path in OBS_CSV.items():
        df = pd.read_csv(csv_path, low_memory=False)
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["source"] = source
        frames.append(df)
    obs = pd.concat(frames, ignore_index=True)
    obs["tidal_basin_name"] = obs["tidal_basin_name"].replace(BASIN_CANONICAL)
    obs = obs.dropna(subset=["date", "x", "y", "tidal_basin_name"]).copy()
    obs["year"] = obs["date"].dt.year
    return obs


def load_grid(spinup_dir: Path) -> dict | None:
    """Static grid fields from the first 3-D file of a run."""
    first = next(iter(sorted(spinup_dir.glob("dws_500m.3d.*.nc"))), None)
    if first is None:
        return None
    with xr.open_dataset(first, mask_and_scale=False) as g:
        lon = g["lonc"].values.astype(float)
        lat = g["latc"].values.astype(float)
        bathy = g["bathymetry"].values.astype(float)
    valid = ((lon > LON_VALID[0]) & (lon < LON_VALID[1])
             & (lat > LAT_VALID[0]) & (lat < LAT_VALID[1]))
    wet = valid & (bathy > BATHY_FILL)
    return {"lon": np.where(valid, lon, np.nan), "lat": np.where(valid, lat, np.nan),
            "depth": np.where(wet, bathy, np.nan), "valid": valid, "wet": wet}


def xy_km(lon, lat, lat0):
    """Local equirectangular coordinates in km."""
    return np.column_stack([np.radians(lon) * np.cos(np.radians(lat0)) * R_EARTH_KM,
                            np.radians(lat) * R_EARTH_KM])


def build_kdtree(grid):
    """KD-tree of wet cell centres, in km."""
    wet = grid["wet"]
    lat0 = float(np.mean(grid["lat"][wet]))
    iy, ix = np.nonzero(wet)
    return cKDTree(xy_km(grid["lon"][wet], grid["lat"][wet], lat0)), iy, ix, lat0


def load_model_year(spinup_dir: Path, year: int) -> xr.Dataset | None:
    """All monthly files of *year*: fill values masked, month boundaries de-duplicated."""
    files = sorted(spinup_dir.glob(NC_PATTERN.format(year=year)))
    if not files:
        return None
    parts = []
    for fp in files:
        with xr.open_dataset(fp, mask_and_scale=False) as dsi:
            missing = [v for v in MODEL_VARS if v not in dsi.data_vars]
            if missing:
                warnings.warn(f"{fp.name}: missing {missing}, file skipped")
                continue
            sub = dsi[MODEL_VARS].load()
        parts.append(sub.where(sub > -1.0))        # -9999 and -9998 -> NaN
    if not parts:
        return None
    ds = xr.concat(parts, dim="time", data_vars="minimal", coords="minimal",
                   compat="override", join="override")
    score = sum(ds[v].notnull().sum(dim=[d for d in ds[v].dims if d != "time"]).values
                for v in MODEL_VARS)
    t = ds["time"].values
    order = np.lexsort((-score, t))                 # time ascending, most complete first
    keep = np.r_[True, t[order][1:] != t[order][:-1]]
    return ds.isel(time=order[keep])


def nearest_time_idx(model_times: pd.DatetimeIndex, obs_times) -> np.ndarray:
    """Index of the nearest model time for each observation time."""
    mt = model_times.to_numpy(dtype="datetime64[ns]")
    ot = np.asarray(obs_times, dtype="datetime64[ns]")
    right = np.clip(np.searchsorted(mt, ot, side="left"), 0, len(mt) - 1)
    left = np.clip(right - 1, 0, len(mt) - 1)
    use_right = np.abs(mt[right] - ot) < np.abs(ot - mt[left])
    return np.where(use_right, right, left)


def model_da_for_var(ds: xr.Dataset, mvar: str) -> xr.DataArray:
    """Model field for *mvar*; suspension feeders = adults (Y3c) + young (Yy3c)."""
    return ds["Y3c"] + ds["Yy3c"] if mvar == "Y3c" else ds[mvar]


def collate_spinup(spinup_name: str, obs: pd.DataFrame, grid: dict) -> pd.DataFrame:
    """Model value at every observation point and date, for one run."""
    spinup_dir = BASE_OUTPUT_DIR / spinup_name
    tree, flat_iy, flat_ix, lat0 = build_kdtree(grid)
    obs = obs.copy()
    dist, nn = tree.query(xy_km(obs["x"].to_numpy(), obs["y"].to_numpy(), lat0))
    obs["iy"], obs["ix"], obs["match_km"] = flat_iy[nn], flat_ix[nn], dist
    inside = dist <= MAX_MATCH_KM
    for mvar in VALIDATION_VARS:
        obs[f"{mvar}_model_gC_m2"] = np.nan
    obs["model_time"] = pd.NaT

    matched, absent = [], []
    for year in sorted(obs.loc[inside, "year"].unique()):
        sel = inside & (obs["year"] == year).to_numpy()
        ds = load_model_year(spinup_dir, int(year))
        if ds is None:
            absent.append(int(year))
            continue
        mt = pd.DatetimeIndex(ds["time"].values)
        o = obs.loc[sel]
        it = nearest_time_idx(mt, o["date"].to_numpy(dtype="datetime64[ns]"))
        at = dict(time=xr.DataArray(it, dims="obs"),
                  yc=xr.DataArray(o["iy"].to_numpy(), dims="obs"),
                  xc=xr.DataArray(o["ix"].to_numpy(), dims="obs"))
        obs.loc[sel, "model_time"] = mt[it]
        for mvar in VALIDATION_VARS:
            obs.loc[sel, f"{mvar}_model_gC_m2"] = (
                model_da_for_var(ds, mvar).isel(**at).values * MGC_TO_GC)
        matched.append(int(year))

    obs["spinup"] = spinup_name
    n_out = int((~inside).sum())
    print(f"  {n_out:,} samples ({100 * n_out / len(obs):.1f}%) are > {MAX_MATCH_KM} km "
          "from a wet cell - outside the domain, left unmatched")
    print(f"  model years matched: {matched or 'none'}")
    if absent:
        print(f"  observation years without model output: {absent[0]}-{absent[-1]} "
              f"({len(absent)} years)")
    return obs

---
## 3. Collocate every run

In [ ]:
obs = load_observations()
print(f"SIBES rows: {(obs['source'] == 'SIBES').sum():,}   "
      f"SUBES rows: {(obs['source'] == 'SUBES').sum():,}   "
      f"dates {obs['date'].min():%Y-%m-%d} .. {obs['date'].max():%Y-%m-%d}")

COLLOC, GRID = {}, None
for spinup in SPINUP_NAMES:
    spinup_dir = BASE_OUTPUT_DIR / spinup
    grid = load_grid(spinup_dir) if spinup_dir.is_dir() else None
    if grid is None:
        print(f"\n{spinup}: no model output found - skipped")
        continue
    GRID = GRID or grid
    csv_path = OUT_DIR / spinup / "benthic_collocated_model_vs_obs.csv"
    print(f"\n{spinup}")
    if REUSE_COLLOCATED and csv_path.exists():
        df = pd.read_csv(csv_path, parse_dates=["date", "model_time"], low_memory=False)
        print(f"  reloaded {csv_path.name}")
    else:
        df = collate_spinup(spinup, obs, grid)
        csv_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(csv_path, index=False)
        print(f"  saved {csv_path}")
    COLLOC[spinup] = df
    n_pairs = {mv: int(df[f"{mv}_model_gC_m2"].notna().sum()) for mv in VALIDATION_VARS}
    print("  matched samples:", ", ".join(f"{k} {v:,}" for k, v in n_pairs.items()))

assert COLLOC, "no collocated data - check BASE_OUTPUT_DIR and SPINUP_NAMES"
RUNS = list(COLLOC)

---
## 4. Skill scores

Per run, feeding group and survey: number of pairs, means, bias
(model − observed), RMSE, Pearson *r* and Spearman *ρ*. Spearman is robust to
the heavy right tail of benthic biomass and is the better guide to whether the
model places high and low biomass in the right stations. `nrmse` is RMSE
divided by the observed standard deviation: above 1, the model does worse at a
station than simply predicting the observed mean everywhere.

In [ ]:
def pairs(df, mvar, source=None):
    """(observed, modelled) arrays of the valid pairs, optionally for one survey."""
    o = df[OBS_COL[mvar]].to_numpy(float)
    m = df[f"{mvar}_model_gC_m2"].to_numpy(float)
    ok = np.isfinite(o) & np.isfinite(m) & (o >= 0) & (m >= 0)
    if source is not None:
        ok &= (df["source"] == source).to_numpy()
    return o[ok], m[ok]


def skill(o, m) -> dict:
    n = len(o)
    out = {"n": n, "obs_mean": np.nan, "mod_mean": np.nan, "obs_std": np.nan,
           "bias": np.nan, "rmse": np.nan, "nrmse": np.nan, "r": np.nan, "rho": np.nan}
    if n < 3:
        return out
    out.update(obs_mean=o.mean(), mod_mean=m.mean(), obs_std=o.std(),
               bias=(m - o).mean(), rmse=np.sqrt(((m - o) ** 2).mean()))
    out["nrmse"] = out["rmse"] / out["obs_std"] if out["obs_std"] > 0 else np.nan
    if np.std(o) > 0 and np.std(m) > 0:
        out["r"] = pearsonr(o, m)[0]
        out["rho"] = spearmanr(o, m)[0]
    return out


rows = []
for run, df in COLLOC.items():
    for mvar in VALIDATION_VARS:
        for source in [None] + SOURCES:
            rows.append({"spinup": run, "variable": mvar, "group": GROUP_LABEL[mvar],
                         "source": source or "all", **skill(*pairs(df, mvar, source))})
SKILL = pd.DataFrame(rows)
SKILL.round(4).to_csv(OUT_DIR / "benthic_skill_summary.csv", index=False)

show = SKILL[SKILL["n"] >= MIN_N].set_index(["variable", "source", "spinup"])
display(show[["n", "obs_mean", "mod_mean", "bias", "rmse", "nrmse", "r", "rho"]].round(3))
thin = SKILL[(SKILL["n"] > 0) & (SKILL["n"] < MIN_N)]
if len(thin):
    print(f"fewer than {MIN_N} pairs (no skill shown): "
          + ", ".join(sorted({f"{r.source} {r.variable} (n={r.n})" for r in thin.itertuples()})))

---
## 5. Figures

Shared plotting helpers: symmetric-log axes, the map background and the
model/observed ratio scale used in Fig. 3.

In [ ]:
def nice_ceiling(v):
    """Round up to 1, 2 or 5 x 10^k."""
    v = max(float(v), LINTHRESH * 10)
    k = np.floor(np.log10(v))
    for m in (1, 2, 5, 10):
        if m * 10 ** k >= v * 0.999:
            return float(m * 10 ** k)


def symlog(ax, axis, vmax, symmetric=False):
    """Symmetric-log axis: linear in [-LINTHRESH, LINTHRESH], decades above.

    On a symmetric (error) axis only 0 and every other decade from
    10 x LINTHRESH are labelled, so labels never collide.
    """
    (ax.set_xscale if axis == "x" else ax.set_yscale)(
        "symlog", linthresh=LINTHRESH, linscale=0.8)
    dec = 10.0 ** np.arange(np.log10(LINTHRESH), np.log10(vmax) + 1e-9)
    minor = np.array([m * d for d in dec for m in range(2, 10) if m * d <= vmax])
    major = np.r_[0.0, dec]
    labelled = list(major)
    if symmetric:
        major = np.r_[-dec[::-1], major]
        minor = np.r_[-minor[::-1], minor]
        keep = [d for d in dec if d >= 10 * LINTHRESH * 0.999
                and round(np.log10(d / (10 * LINTHRESH))) % 2 == 0]
        labelled = [0.0] + keep + [-k for k in keep]
    a = ax.xaxis if axis == "x" else ax.yaxis
    a.set_major_locator(mticker.FixedLocator(major))
    a.set_minor_locator(mticker.FixedLocator(minor))
    a.set_major_formatter(mticker.FuncFormatter(
        lambda v, _: ("0" if v == 0 else fs.unsigned(v, "g"))
        if np.any(np.isclose(v, labelled, rtol=1e-6, atol=0)) else ""))


def binned_median(o, m, min_n=20):
    """Median model value in half-decade bins of the observation (zeros on their own)."""
    rows = []
    z = o == 0
    if z.sum() >= min_n:
        rows.append((0.0, np.median(m[z]), int(z.sum())))
    edges = 10.0 ** np.arange(np.log10(LINTHRESH), 4.01, 0.5)
    lo = 0.0
    for hi in edges:
        sel = (o > lo) & (o <= hi)
        if sel.sum() >= min_n:
            rows.append((np.median(o[sel]), np.median(m[sel]), int(sel.sum())))
        lo = hi
    return pd.DataFrame(rows, columns=["obs_median", "model_median", "n"])


def group_title(mvar):
    return GROUP_LABEL[mvar]


def source_label(src):
    """Legend label with the matched sample count when it is the same everywhere."""
    ns = {len(pairs(df, mv, src)[0]) for df in COLLOC.values() for mv in VALIDATION_VARS}
    return f"{src} (n = {ns.pop():,})" if len(ns) == 1 else src


# --- map background, shared by every map -------------------------------------
_matched = pd.concat([df.loc[df[[f"{v}_model_gC_m2" for v in VALIDATION_VARS]].notna().any(axis=1)]
                      for df in COLLOC.values()])
MAP_EXTENT = (float(_matched["x"].min()) - 0.07, float(_matched["x"].max()) + 0.07,
              float(_matched["y"].min()) - 0.06, float(_matched["y"].max()) + 0.07)
LAT0 = 0.5 * (MAP_EXTENT[2] + MAP_EXTENT[3])
LAND = fs.load_land(MAP_EXTENT)
LON_F = fs.fill_coord(GRID["lon"], "lonc (for plotting)", verbose=False)
LAT_F = fs.fill_coord(GRID["lat"], "latc (for plotting)", verbose=False)


def map_background(ax, run, labels=(True, True)):
    fs.setup_map(ax, MAP_EXTENT, lat0=LAT0, xstep=0.5, ystep=0.2, labels=labels)
    fs.add_bathymetry(ax, LON_F, LAT_F, GRID["depth"])
    fs.add_land(ax, LAND)
    fs.add_domain_outline(ax, GRID["lon"], GRID["lat"], SPINUP_COLOURS[run],
                          valid=GRID["valid"])
    fs.add_water_label(ax, 5.62, 53.555, "North Sea")


# --- model/observed ratio as a bounded, symmetric quantity -------------------
# d = (m - o) / (m + o) lies in [-1, 1]; it is 0 for a perfect match, -1 when the
# model is zero and +1 when the observation is zero, and m/o = (1 + d) / (1 - d).
RATIO_TICKS = [0, 1 / 10, 1 / 3, 1, 3, 10, np.inf]
RATIO_TICK_POS = [-1.0] + [(r - 1) / (r + 1) for r in RATIO_TICKS[1:-1]] + [1.0]
RATIO_TICK_LAB = ["0", "1/10", "1/3", "1", "3", "10", "∞"]
print("map extent:", tuple(round(v, 2) for v in MAP_EXTENT))

### Fig. 1 — modelled vs observed biomass

One column per feeding group, one row per run. Points are individual samples
matched in space and time; the dashed line is 1:1 and the grey line the median
modelled biomass in half-decade bins of the observation (observed zeros form
their own bin). Scores (bias and RMSE in g C m⁻², Pearson *r*) use all surveys
together.

In [ ]:
def fig_scatter():
    n_r, n_g = len(RUNS), len(VALIDATION_VARS)
    fig, axes = fs.figure("double", 12 + 45 * n_r, nrows=n_r, ncols=n_g, squeeze=False)
    csv, letters = [], iter("abcdefghijklmnopqrstuvwxyz")
    for g, mvar in enumerate(VALIDATION_VARS):
        allv = np.concatenate([np.concatenate(pairs(df, mvar)) for df in COLLOC.values()])
        vmax = nice_ceiling(allv.max()) if allv.size else 1.0
        for r, run in enumerate(RUNS):
            ax, df = axes[r, g], COLLOC[run]
            for src in SOURCES:
                o, m = pairs(df, mvar, src)
                if not len(o):
                    continue
                big = src == "SUBES"
                ax.scatter(o, m, s=7 if big else 2.5, marker=fs.SOURCE_MARKERS[src],
                           color=fs.SOURCE_COLOURS[src], alpha=0.9 if big else 0.35,
                           linewidths=0, rasterized=True, zorder=3 if big else 2)
                csv.append(pd.DataFrame({"spinup": run, "variable": mvar, "source": src,
                                         "kind": "sample", "observed": o, "modelled": m}))
            o, m = pairs(df, mvar)
            ax.plot([0, vmax], [0, vmax], ls=(0, (4, 2)), lw=0.6, color=fs.INK, zorder=4)
            bm = binned_median(o, m)
            ax.plot(bm["obs_median"], bm["model_median"], color=fs.INK2, lw=1.0,
                    marker="o", ms=2.2, zorder=5)
            csv.append(bm.rename(columns={"obs_median": "observed", "model_median": "modelled"})
                       .assign(spinup=run, variable=mvar, source="all", kind="binned median"))
            for axis in "xy":
                symlog(ax, axis, vmax)
            lo = -0.4 * LINTHRESH
            ax.set_xlim(lo, vmax); ax.set_ylim(lo, vmax); ax.set_box_aspect(1)
            s = skill(o, m)
            if s["n"] >= MIN_N:
                fs.stats_line(ax, f"bias {fs.signed(s['bias'])} \u00b7 RMSE {s['rmse']:.2f}"
                                  f" \u00b7 r {fs.unsigned(s['r'])}")
            if r == 0:
                ax.set_title(group_title(mvar), pad=10)
            if r == n_r - 1:
                ax.set_xlabel(f"Observed ({UNIT})")
            if g == 0:
                ax.set_ylabel(f"Modelled ({UNIT})")
            if g == n_g - 1:
                ax.annotate(run, xy=(1, 0.5), xycoords="axes fraction", xytext=(4, 0),
                            textcoords="offset points", rotation=270, ha="left",
                            va="center", fontweight="bold", color=SPINUP_COLOURS[run])
    handles = [Line2D([], [], ls="", marker=fs.SOURCE_MARKERS[s], ms=3.5,
                      color=fs.SOURCE_COLOURS[s], label=source_label(s)) for s in SOURCES]
    handles += [Line2D([], [], ls=(0, (4, 2)), lw=0.6, color=fs.INK, label="1:1"),
                Line2D([], [], color=fs.INK2, lw=1.0, marker="o", ms=2.2,
                       label="binned median")]
    fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
    for ax in axes.flat:
        fs.panel_label(ax, next(letters))
    fs.save_figure(fig, "fig01_benthos_scatter", FIG_DIR, data=pd.concat(csv, ignore_index=True))
    return fig


fig_scatter();

### Fig. 2 — distribution of model error

Fraction of samples per bin of model − observed biomass, per survey. Bins are
equal in width on the symmetric-log axis, so the curve is a density in the
plotted coordinate. The triangle marks the median error; the dashed line is
zero. A survey with fewer than `MIN_N` pairs is not drawn.

In [ ]:
def error_edges(xmax):
    e = 10.0 ** np.arange(np.log10(LINTHRESH), np.log10(xmax) + 1e-9, 0.2)
    inner = np.linspace(-LINTHRESH, LINTHRESH, 5)[1:-1]
    return np.r_[-e[::-1], inner, e]


def fig_error_distribution():
    n_r, n_g = len(RUNS), len(VALIDATION_VARS)
    fig, axes = fs.figure("double", 14 + 36 * n_r, nrows=n_r, ncols=n_g, squeeze=False)
    csv, letters, shown = [], iter("abcdefghijklmnopqrstuvwxyz"), set()
    for g, mvar in enumerate(VALIDATION_VARS):
        errs = np.concatenate([np.subtract(*pairs(df, mvar)[::-1]) for df in COLLOC.values()])
        xmax = nice_ceiling(np.percentile(np.abs(errs), 99.8)) if errs.size else 1.0
        edges = error_edges(xmax)
        for r, run in enumerate(RUNS):
            ax, df = axes[r, g], COLLOC[run]
            notes = []
            for src in SOURCES:
                o, m = pairs(df, mvar, src)
                e = np.clip(m - o, edges[0], edges[-1])
                if len(e) < MIN_N:
                    continue
                shown.add(src)
                frac = np.histogram(e, edges)[0] / len(e)
                c = fs.SOURCE_COLOURS[src]
                ax.stairs(frac, edges, color=c, lw=0.9, zorder=3)
                ax.stairs(frac, edges, color=c, alpha=0.15, fill=True, lw=0, zorder=2)
                med = float(np.median(m - o))
                ax.plot([med], [1.0], marker="v", ms=3.2, color=c, clip_on=False,
                        transform=ax.get_xaxis_transform(), zorder=5)
                notes.append((src, f"median {fs.signed(med)} \u00b7 "
                                   f"bias {fs.signed(np.mean(m - o))}"))
                csv.append(pd.DataFrame({"spinup": run, "variable": mvar, "source": src,
                                         "bin_lo": edges[:-1], "bin_hi": edges[1:],
                                         "fraction": frac}))
            ax.axvline(0, color=fs.INK, lw=0.5, ls=(0, (4, 2)), zorder=4)
            symlog(ax, "x", xmax, symmetric=True)
            ax.set_xlim(-xmax, xmax)
            ax.set_ylim(0, None)
            ax.margins(y=0.12)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
            if notes:
                fs.stats_line(ax, "\n".join(t if len(notes) == 1 else f"{src_}: {t}"
                                            for src_, t in notes), pad=6)
            if r == 0:
                ax.set_title(group_title(mvar), pad=6 + 8 * max(len(notes), 1))
            if r == n_r - 1:
                ax.set_xlabel(f"Model − observed ({UNIT})")
            if g == 0:
                ax.set_ylabel("Fraction of samples")
            if g == n_g - 1:
                ax.annotate(run, xy=(1, 0.5), xycoords="axes fraction", xytext=(4, 0),
                            textcoords="offset points", rotation=270, ha="left",
                            va="center", fontweight="bold", color=SPINUP_COLOURS[run])
    handles = [Line2D([], [], color=fs.SOURCE_COLOURS[s], lw=0.9,
                      label=source_label(s) if s in shown
                      else f"{source_label(s)}: fewer than {MIN_N} pairs, not shown")
               for s in SOURCES]
    fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
    for ax in axes.flat:
        fs.panel_label(ax, next(letters))
    fs.save_figure(fig, "fig02_benthos_error_distribution", FIG_DIR,
                   data=pd.concat(csv, ignore_index=True) if csv else None)
    return fig


fig_error_distribution();

### Fig. 3 — where the model is too high or too low

One figure per feeding group. Top row: station bias, model − observed
(g C m⁻²), on a symmetric scale clipped at the 98th percentile of |bias|
(arrows mark values beyond it). Bottom row: model/observed ratio. Where a
station was sampled more than once, model and observation are averaged first.
Circles are SIBES, outlined squares SUBES. The coloured outline is the model
domain; samples beyond its wet cells are not matched.

The ratio replaces the earlier relative error (model − obs)/model, which is
capped at +1 for over-prediction but unbounded for under-prediction, so it
could not be shown on a symmetric scale. Here the ratio is drawn through the
normalised difference (m − o)/(m + o): symmetric, bounded, defined where the
observation is zero, and labelled on the colour bar in ratios (1/10 … 10; the
ends are "model zero" and "observed zero").

In [ ]:
def station_means(df, mvar):
    ocol, mcol = OBS_COL[mvar], f"{mvar}_model_gC_m2"
    ok = df[ocol].notna() & df[mcol].notna() & (df[ocol] >= 0) & (df[mcol] >= 0)
    g = (df[ok].groupby(["source", "sampling_station_id"])
         .agg(x=("x", "mean"), y=("y", "mean"), observed=(ocol, "mean"),
              modelled=(mcol, "mean"), n_samples=(ocol, "size")).reset_index())
    g["bias"] = g["modelled"] - g["observed"]
    s = g["modelled"] + g["observed"]
    g["norm_diff"] = np.where(s > 0, g["bias"] / s.where(s > 0, 1.0), 0.0)
    return g


def fig_bias_map(mvar):
    n_c = len(RUNS)
    means = {run: station_means(COLLOC[run], mvar) for run in RUNS}
    lim = fs.nice_limit(pd.concat(means.values())["bias"], q=98)
    # biomass bias is as skewed as biomass: a symmetric-log norm keeps the many
    # small biases visible next to the few large ones
    lin = 10.0 ** (np.floor(np.log10(lim)) - 2)
    norms = {"bias": mcolors.SymLogNorm(linthresh=lin, linscale=1.0, vmin=-lim, vmax=lim,
                                        base=10),
             "norm_diff": mcolors.Normalize(-1.0, 1.0)}
    pos = [10.0 ** k for k in range(int(round(np.log10(lin))), int(np.floor(np.log10(lim))) + 1)
           if lim / 10.0 ** k >= 3]
    bias_ticks = [-lim] + [-p for p in pos[::-1]] + [0.0] + pos + [lim]
    cmap = fs.cmap_diverging()
    # panel size from the map aspect so that the figure has no dead space
    w_panel = (183 - 24) / n_c
    h_panel = w_panel * (MAP_EXTENT[3] - MAP_EXTENT[2]) / (
        (MAP_EXTENT[1] - MAP_EXTENT[0]) * np.cos(np.radians(LAT0)))
    fig, axes = fs.figure("double", 2 * h_panel + 22, nrows=2, ncols=n_c, squeeze=False)
    letters = iter("abcdefghijklmnopqrstuvwxyz")
    mappables = {}
    for c, run in enumerate(RUNS):
        g = means[run]
        for r, metric in enumerate(["bias", "norm_diff"]):
            ax = axes[r, c]
            map_background(ax, run, labels=(r == 1, c == 0))
            for src in SOURCES:
                d = g[g["source"] == src]
                if d.empty:
                    continue
                big = src == "SUBES"
                sc = ax.scatter(d["x"], d["y"], c=d[metric], cmap=cmap, norm=norms[metric],
                                s=7 if big else 1.4, marker=fs.SOURCE_MARKERS[src],
                                edgecolors=fs.INK if big else "none",
                                linewidths=0.3 if big else 0, rasterized=True,
                                zorder=6 if big else 5)
                mappables[metric] = sc
            if r == 0:
                ax.set_title(run, color=SPINUP_COLOURS[run], fontweight="bold")
                counts = g["source"].value_counts()
                fs.corner_note(ax, "  ".join(f"{s} {counts.get(s, 0):,}" for s in SOURCES),
                               corners=("lower left",), fontsize=6, color=fs.INK2,
                               path_effects=fs.halo())
            if r == 1 and c == n_c - 1:
                fs.add_scalebar(ax, 20, "lower right")
    fs.corner_note(axes[0, 0], group_title(mvar), corners=("upper left",),
                   fontsize=7, fontweight="bold", path_effects=fs.halo())
    if "bias" in mappables:
        fs.diverging_colorbar(fig, mappables["bias"], ax=axes[0, :], ticks=bias_ticks,
                              ticklabels=[fs.unsigned(t, "g") for t in bias_ticks],
                              label=f"Model − observed ({UNIT})", extend="both",
                              shrink=0.9, aspect=18, pad=0.02)
    if "norm_diff" in mappables:
        fs.diverging_colorbar(fig, mappables["norm_diff"], ax=axes[1, :],
                              ticks=RATIO_TICK_POS, ticklabels=RATIO_TICK_LAB,
                              label="Model / observed", shrink=0.9, aspect=18, pad=0.02)
    for ax in axes.flat:
        fs.panel_label(ax, next(letters))
    data = pd.concat({run: means[run] for run in RUNS}, names=["spinup"]).reset_index(level=0)
    fs.save_figure(fig, f"fig03_benthos_bias_map_{mvar}", FIG_DIR, data=data)
    return fig


for mvar in VALIDATION_VARS:
    fig_bias_map(mvar)

### Fig. 4 — skill summary across runs

For every feeding group and survey with at least `MIN_N` pairs: (a) mean
biomass, observed (open) and modelled (filled, one colour per run); (b) Pearson
*r*; (c) RMSE normalised by the observed standard deviation — right of the
dashed line at 1 the model does worse than the observed mean would.

In [ ]:
def fig_skill_summary():
    tab = SKILL[(SKILL["source"] != "all") & (SKILL["n"] >= MIN_N)]
    keys = [(mv, s) for mv in VALIDATION_VARS for s in SOURCES
            if ((tab["variable"] == mv) & (tab["source"] == s)).any()]
    if not keys:
        print("no feeding group has enough pairs for a skill summary")
        return None
    fig, axes = fs.figure("double", 22 + 8 * len(keys), ncols=3, sharey=True)
    y = np.arange(len(keys))[::-1].astype(float)
    off = np.linspace(-0.16, 0.16, len(RUNS)) if len(RUNS) > 1 else [0.0]
    for yi, (mv, s) in zip(y, keys):
        t = tab[(tab["variable"] == mv) & (tab["source"] == s)].set_index("spinup")
        axes[0].plot(t["obs_mean"].iloc[0], yi, "o", ms=4, mfc="white", mec=fs.INK,
                     mew=0.8, zorder=4)
        for k, run in enumerate(RUNS):
            if run not in t.index:
                continue
            kw = dict(ls="", marker="o", ms=3.6, color=SPINUP_COLOURS[run], zorder=5)
            axes[0].plot(t.loc[run, "mod_mean"], yi + off[k], **kw)
            axes[1].plot(t.loc[run, "r"], yi + off[k], **kw)
            axes[2].plot(t.loc[run, "nrmse"], yi + off[k], **kw)
    axes[0].set_xscale("log")
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
    axes[0].set_xlabel(f"Mean biomass ({UNIT})")
    axes[1].axvline(0, color=fs.INK3, lw=0.5, ls=(0, (4, 2)))
    axes[1].set_xlabel("Pearson r")
    r_abs = np.nanmax(np.abs(tab["r"].to_numpy(float)))
    r_lim = max(0.25, np.ceil(r_abs * 1.15 / 0.05) * 0.05)
    axes[1].set_xlim(-r_lim, r_lim)
    axes[2].axvline(1, color=fs.INK3, lw=0.5, ls=(0, (4, 2)))
    axes[2].set_xlim(0, max(2.0, np.nanmax(tab["nrmse"].to_numpy(float)) * 1.1))
    axes[2].set_xlabel(r"RMSE / $\sigma_{\mathrm{obs}}$")
    labels = [GROUP_LABEL[mv] + ("" if len({k[1] for k in keys}) == 1 else f" ({s})")
              for mv, s in keys]
    axes[0].set_yticks(y, labels)
    axes[0].tick_params(axis="y", which="both", length=0)
    axes[0].set_ylim(y.min() - 0.6, y.max() + 0.6)
    for ax in axes:
        ax.yaxis.set_minor_locator(mticker.NullLocator())
        for sp in ("left",):
            ax.spines[sp].set_visible(False)
    for ax in axes[1:]:
        ax.tick_params(axis="y", which="both", length=0)
    handles = [Line2D([], [], ls="", marker="o", ms=4, mfc="white", mec=fs.INK,
                      mew=0.8, label="observed")]
    handles += [Line2D([], [], ls="", marker="o", ms=3.6, color=SPINUP_COLOURS[r], label=r)
                for r in RUNS]
    fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
    for ax, l in zip(axes, "abc"):
        fs.panel_label(ax, l)
    fs.save_figure(fig, "fig04_benthos_skill_summary", FIG_DIR,
                   data=tab[["spinup", "variable", "group", "source", "n", "obs_mean",
                             "mod_mean", "bias", "rmse", "nrmse", "r", "rho"]])
    return fig


fig_skill_summary();

### Fig. 5 — observed and modelled distributions

Cumulative distribution of biomass over the matched samples (all surveys).
Where the observed curve rises steeply at zero, most samples held none of that
group; a model curve to the right of the observed one over-predicts, to the
left under-predicts. A model that is narrower than the observations
(steeper curve) misses the patchiness — typical of a model resolving
500 m cells against point samples.

In [ ]:
def ecdf(v):
    v = np.sort(v)
    return v, np.arange(1, v.size + 1) / v.size


def fig_distribution():
    fig, axes = fs.figure("onehalf", 100, nrows=2, ncols=2, squeeze=False)
    csv = []
    for ax, mvar, l in zip(axes.flat, VALIDATION_VARS, "abcd"):
        o, _ = pairs(COLLOC[RUNS[0]], mvar)
        allv = np.concatenate([np.concatenate(pairs(df, mvar)) for df in COLLOC.values()])
        vmax = nice_ceiling(allv.max()) if allv.size else 1.0
        series = [("observed", o, fs.OBS_COLOUR, 1.2)]
        series += [(run, pairs(COLLOC[run], mvar)[1], SPINUP_COLOURS[run], 1.0) for run in RUNS]
        for name, v, c, lw in series:
            if not v.size:
                continue
            x, p = ecdf(v)
            ax.step(np.r_[x[0], x], np.r_[0, p], where="post", color=c, lw=lw,
                    zorder=4 if name == "observed" else 3)
            csv.append(pd.DataFrame({"variable": mvar, "series": name,
                                     "biomass": x, "cum_fraction": p}))
        symlog(ax, "x", vmax)
        ax.set_xlim(-0.4 * LINTHRESH, vmax)
        ax.set_ylim(0, 1.0)
        ax.yaxis.set_major_locator(mticker.MultipleLocator(0.5))
        ax.set_title(group_title(mvar))
        if ax in axes[-1]:
            ax.set_xlabel(f"Biomass ({UNIT})")
        if ax in axes[:, 0]:
            ax.set_ylabel("Cumulative fraction")
        fs.panel_label(ax, l)
    handles = [Line2D([], [], color=fs.OBS_COLOUR, lw=1.2, label="observed")]
    handles += [Line2D([], [], color=SPINUP_COLOURS[r], lw=1.0, label=r) for r in RUNS]
    fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
    fs.save_figure(fig, "fig05_benthos_distribution", FIG_DIR,
                   data=pd.concat(csv, ignore_index=True))
    return fig


fig_distribution();

### Fig. 6 — total biomass and community composition by depth

Samples are grouped by the depth below MSL of the model cell they are matched
to, taken from the model `bathymetry` (its reference level is treated as MSL;
negative depths are beds above MSL, i.e. the higher flats). Only samples with
all four feeding groups observed and modelled enter.

(a) Mean total biomass (sum of the four groups), observed (open) and modelled
(filled), with 95 % bootstrap intervals of the mean. (b onwards) Share of each
feeding group in the class-mean total biomass, observed and per run; numbers
are percentages. Shares are ratios of class means, so a group that is absent
from most samples still counts with its mean biomass. Classes with fewer than
`MIN_N` samples are omitted, and the sample count of each class is given under
its label.

In [ ]:
def depth_label(lo, hi):
    f = lambda v: fs.unsigned(v, "g")
    if not np.isfinite(lo):
        return f"≤ {f(hi)}"
    if not np.isfinite(hi):
        return f"> {f(lo)}"
    return f"{f(lo)} to {f(hi)}"


DEPTH_LABELS = [depth_label(a, b) for a, b in zip(DEPTH_EDGES[:-1], DEPTH_EDGES[1:])]


def depth_class_table(n_boot=2000, seed=0):
    """Mean biomass per feeding group, total and shares, per depth class and series."""
    rng = np.random.default_rng(seed)
    ocols = [OBS_COL[v] for v in VALIDATION_VARS]
    rows = []
    for k, run in enumerate(RUNS):
        df = COLLOC[run]
        mcols = [f"{v}_model_gC_m2" for v in VALIDATION_VARS]
        ok = (df[ocols + mcols].notna().all(axis=1)
              & (df[ocols] >= 0).all(axis=1) & (df[mcols] >= 0).all(axis=1))
        d = df.loc[ok].copy()
        d["model_depth_m"] = GRID["depth"][d["iy"].astype(int), d["ix"].astype(int)]
        d["depth_class"] = pd.cut(d["model_depth_m"], DEPTH_EDGES, labels=DEPTH_LABELS)
        series = ([("observed", ocols)] if k == 0 else []) + [(run, mcols)]
        for name, cols in series:
            for cls, g in d.groupby("depth_class", observed=True):
                vals = g[cols].to_numpy(float)
                total = vals.sum(axis=1)
                means = vals.mean(axis=0)
                idx = rng.integers(0, len(total), size=(n_boot, len(total)))
                boot = total[idx].mean(axis=1)
                row = {"series": name, "depth_class": str(cls), "n": len(g),
                       "depth_median_m": float(g["model_depth_m"].median()),
                       "total_mean": total.mean(), "total_lo": np.percentile(boot, 2.5),
                       "total_hi": np.percentile(boot, 97.5)}
                for v, m_ in zip(VALIDATION_VARS, means):
                    row[f"{v}_mean"] = m_
                    row[f"{v}_share_pct"] = 100 * m_ / means.sum() if means.sum() > 0 else np.nan
                rows.append(row)
    return pd.DataFrame(rows)


def _text_colour(c):
    r, g, b = mcolors.to_rgb(c)
    return fs.INK if 0.2126 * r + 0.7152 * g + 0.0722 * b > 0.5 else "white"


def fig_depth_classes():
    tab = depth_class_table()
    n_obs = tab[tab["series"] == "observed"].set_index("depth_class")["n"]
    classes = [c for c in DEPTH_LABELS if n_obs.get(c, 0) >= MIN_N]
    tab = tab[tab["depth_class"].isin(classes)]
    series = ["observed"] + RUNS
    colours = {"observed": fs.OBS_COLOUR, **SPINUP_COLOURS}
    y = np.arange(len(classes), dtype=float)
    fig, axes = fs.figure("double", 34 + 8.5 * len(classes), ncols=1 + len(series), sharey=True,
                          width_ratios=[1.4] + [1] * len(series))

    # (a) total biomass per class
    ax = axes[0]
    off = np.linspace(-0.2, 0.2, len(series)) if len(series) > 1 else [0.0]
    for k, s_ in enumerate(series):
        t = tab[tab["series"] == s_].set_index("depth_class").reindex(classes)
        c = colours[s_]
        ax.errorbar(t["total_mean"], y + off[k],
                    xerr=[t["total_mean"] - t["total_lo"], t["total_hi"] - t["total_mean"]],
                    fmt="o", ms=3.6, color=c, mfc="white" if s_ == "observed" else c, mec=c,
                    mew=0.8, elinewidth=0.8, capsize=0, zorder=3)
    ax.set_xlim(0, None)
    ax.set_xlabel(f"Total biomass ({UNIT})")
    ax.set_ylabel("Depth below MSL (m)")
    ax.set_yticks(y, [f"{c}\nn = {int(n_obs[c]):,}" for c in classes])
    ax.set_ylim(len(classes) - 0.5, -0.5)            # shallowest class on top
    ax.yaxis.set_minor_locator(mticker.NullLocator())

    # (b ...) composition per series
    for j, s_ in enumerate(series, start=1):
        ax = axes[j]
        t = tab[tab["series"] == s_].set_index("depth_class").reindex(classes)
        left = np.zeros(len(classes))
        for v in VALIDATION_VARS:
            w = t[f"{v}_share_pct"].to_numpy(float)
            ax.barh(y, w, left=left, height=0.64, color=fs.GROUP_COLOURS[v],
                    edgecolor="white", linewidth=0.4, zorder=2)
            for yi, l0, wi in zip(y, left, w):
                if np.isfinite(wi) and wi >= 12:
                    ax.text(l0 + wi / 2, yi, f"{wi:.0f}", ha="center", va="center",
                            fontsize=5.5, color=_text_colour(fs.GROUP_COLOURS[v]), zorder=3)
            left += np.nan_to_num(w)
        ax.set_xlim(0, 100)
        ax.xaxis.set_major_locator(mticker.MultipleLocator(50))
        ax.xaxis.set_minor_locator(mticker.MultipleLocator(25))
        ax.set_xlabel("Share of total (%)")
        ax.tick_params(axis="y", which="both", length=0)
        ax.spines["left"].set_visible(False)
        ax.set_title("Observed" if s_ == "observed" else s_, color=colours[s_],
                     fontweight="normal" if s_ == "observed" else "bold")
    axes[0].set_title("Total biomass")

    handles = [Line2D([], [], ls="", marker="o", ms=3.6, mfc="white", mec=fs.OBS_COLOUR,
                      mew=0.8, label="observed")]
    handles += [Line2D([], [], ls="", marker="o", ms=3.6, color=SPINUP_COLOURS[r], label=r)
                for r in RUNS]
    handles += [mpatches.Patch(facecolor=fs.GROUP_COLOURS[v], edgecolor="white",
                               label=GROUP_LABEL[v]) for v in VALIDATION_VARS]
    fig.legend(handles=handles, loc="outside upper center", ncol=len(handles))
    for ax, l in zip(axes, "abcdefgh"):
        fs.panel_label(ax, l)
    fs.save_figure(fig, "fig06_benthos_depth_classes", FIG_DIR, data=tab)
    return tab


DEPTH_TABLE = fig_depth_classes()
display(DEPTH_TABLE.pivot_table(index="depth_class", columns="series", values="total_mean",
                                sort=False).round(2))

---
## 6. Output index

In [ ]:
print("written to", FIG_DIR)
for f in sorted(FIG_DIR.glob("fig*")):
    print(f"   {f.name:<48} {f.stat().st_size / 1024:>8.0f} kB")